# GEE Pre-processing: Friction Surface Data Acquisition

This notebook pulls geospatial data from Google Earth Engine (GEE) and other
sources to processes and export it. The exported rasters are aligned to each other, clipped to Alaska, and share a common CRS.

**All outputs are in EPSG:3413** (WGS 84 / NSIDC Sea Ice Polar Stereographic North),
clipped to Alaska, at **150m resolution** with consistent pixel alignment.

### Output Rasters
| File | Source | Description |
|------|--------|-------------|
| `lulc_alaska_modal.tif` | Dynamic World | Modal land cover classification (classes 0-8) |
| `slope_alaska.tif` | FabDEM | Slope in degrees |
| `dem_alaska.tif` | FabDEM | Raw elevation values |
| `permafrost_alaska.tif` | Pastick et al. 2015 | Permafrost zones (0-3), 30m native resolution |
| `roads_presence_alaska.tif` | GRIP4 | Binary road presence |
| `rivers_alaska.tif` | NHD + USACE NWN | Navigable waterways with true polygon width |
| `sea_ice_concentration_alaska.tif` | NOAA/NSIDC CDR v4 | Climatological winter sea ice concentration (0-100%) |
| `regions_alaska.tif` | AEA Library | Energy region boundaries (integer IDs 1-10) |

### Output Vectors
| File | Source | Description |
|------|--------|-------------|
| `airports_alaska.geojson` | OurAirports | Airport point locations (large/medium/small/heliports) |
| `ports_alaska.geojson` | AK DOT&PF | Port locations with type |
| `facilities_alaska.geojson` | Bulk Fuel CSV | Fuel facility sites |

**References:**
- Trochim et al. (review) -- friction surface methodology
- Atkinson et al., 2005 -- slope classification
- Pastick et al., 2015 -- near-surface permafrost probability (30m)
  DOI: 10.5066/F7C53HX6
- USGS NHD (sat-io GEE catalog) -- water body polygon geometry
- USACE National Waterway Network -- navigability authority
- OurAirports (davidmegginson.github.io/ourairports-data) -- airport locations
- NOAA/NSIDC CDR v4 (G02202) -- passive microwave sea ice concentration

## 1. Setup & Authentication

In [ ]:
# --- Installations ---
#!pip install earthengine-api geopandas rasterio pyproj

# Importa
import ee
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, mapping
import rasterio
from rasterio.features import rasterize as rio_rasterize
from rasterio.crs import CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask as rio_mask
import zipfile
import urllib.request
import shutil

# Authenticate and initialize GEE
ee.Authenticate()
ee.Initialize(project='gee-friction-layer-processing')  # <-- Julia's GEE cloud project name

# Constants
# -------------------
TARGET_CRS = 'EPSG:3413'   # CRD --> NSIDC Sea Ice Polar Stereographic North
TARGET_SCALE = 150          # resolution (meters)
DRIVE_FOLDER = 'friction_surface_exports'  # Google Drive folder for exports

# Alaska bounding box (generous, in WGS84)
ALASKA_BBOX = ee.Geometry.Rectangle([-180, 51, -129, 72])

# Alaska state boundary from TIGER/Census for precise clipping
alaska_states = ee.FeatureCollection('TIGER/2018/States')
alaska_boundary = alaska_states.filter(ee.Filter.eq('NAME', 'Alaska')).geometry()

# Shared CRS transform for pixel alignment across all exports.
CRS_TRANSFORM = [150, 0, -3000000, 0, -150, 3000000]

print(f"Target CRS: {TARGET_CRS}")
print(f"Target scale: {TARGET_SCALE}m")
print(f"Drive folder: {DRIVE_FOLDER}")
print("GEE initialized successfully.")

In [ ]:
#Print current working directory
os.getcwd()

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Change CWD
os.chdir('/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/notebooks/')

# Point RASTER_DIR at the Google Drive folder where the GEE-exported rasters (friction_surface)exports
os.environ['RASTER_DIR'] = '/content/drive/My Drive/friction_surface_exports'

In [ ]:
# Export properly aligned rasters after processing to friction_surfaces_folder

def export_aligned_raster(image, description, filename, region=None,
                          scale=TARGET_SCALE, crs=TARGET_CRS,
                          crs_transform=CRS_TRANSFORM, folder=DRIVE_FOLDER,
                          max_pixels=1e10):
    """Export a GEE image to Google Drive with consistent alignment."""
    if region is None:
        region = alaska_boundary

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=filename,
        region=region,
        crs=crs,
        crsTransform=crs_transform,
        maxPixels=int(max_pixels),
        fileFormat='GeoTIFF',
    )
    task.start()
    print(f"Export started: {description} -> {folder}/{filename}.tif")
    return task


# Collect all tasks for monitoring
export_tasks = []

In [ ]:
# Helper functions for raster processing
# These are reused by the local raster cells (permafrost, NWN waterways)
# to reduce duplication and make the cells clearer.

# Cache for Alaska boundary (computed once, reused everywhere)
_alaska_boundary_3413 = None


def get_alaska_boundary_3413():
    """Return the Alaska state boundary as a GeoDataFrame in EPSG:3413.

    Tries the GEE FeatureCollection first (already loaded in setup),
    falls back to downloading the Census TIGER shapefile if GEE is
    unavailable. Cached so it only fetches once per session.
    """
    global _alaska_boundary_3413
    if _alaska_boundary_3413 is not None:
        return _alaska_boundary_3413

    try:
        ak_geom_info = alaska_boundary.getInfo()
        gdf = gpd.GeoDataFrame.from_features([{
            "type": "Feature", "geometry": ak_geom_info, "properties": {}
        }], crs="EPSG:4326").to_crs("EPSG:3413")
        print("Alaska boundary sourced from GEE.")
    except Exception:
        print("GEE boundary unavailable -- downloading Census TIGER shapefile...")
        states_url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_20m.zip"
        urllib.request.urlretrieve(states_url, "/content/states.zip")
        with zipfile.ZipFile("/content/states.zip", "r") as z:
            z.extractall("/content/states")
        states = gpd.read_file("/content/states/cb_2018_us_state_20m.shp")
        gdf = states[states["NAME"] == "Alaska"].to_crs("EPSG:3413")

    _alaska_boundary_3413 = gdf
    return gdf


def reproject_raster_to_3413(src_path, out_path,
                              resampling=Resampling.bilinear, nodata=-9999):
    """Reproject any raster to EPSG:3413 using rasterio.

    Always outputs as GeoTIFF float32 regardless of input format/dtype.
    """
    dst_crs = CRS.from_epsg(3413)
    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        profile = src.profile.copy()
        profile.update(crs=dst_crs, transform=transform, width=width,
                       height=height, nodata=nodata,
                       driver='GTiff', dtype='float32')
        with rasterio.open(out_path, 'w', **profile) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=transform, dst_crs=dst_crs,
                resampling=resampling,
            )
    return out_path


def clip_raster_to_alaska(src_path, out_path, nodata=-9999):
    """Clip an EPSG:3413 raster to the Alaska state boundary.

    Always outputs as GeoTIFF float32 regardless of input format/dtype.
    """
    ak_gdf = get_alaska_boundary_3413()
    ak_shapes = [mapping(geom) for geom in ak_gdf.geometry]

    with rasterio.open(src_path) as src:
        out_image, out_transform = rio_mask(src, ak_shapes, crop=True, nodata=nodata)
        profile = src.profile.copy()
        profile.update(transform=out_transform,
                       height=out_image.shape[1],
                       width=out_image.shape[2],
                       nodata=nodata,
                       driver='GTiff', dtype='float32')
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(out_image.astype(np.float32))
    return out_path


def resample_to_reference_grid(src_path, ref_path, out_path,
                                dtype='int8', resampling=Resampling.nearest,
                                nodata=-1):
    """Resample a source raster to match a reference raster's grid.

    Use Resampling.nearest for categorical data (zone IDs, classes),
    Resampling.bilinear for continuous data (elevation, slope).
    """
    with rasterio.open(ref_path) as ref:
        ref_profile = ref.profile.copy()
        ref_transform = ref.transform
        ref_crs = ref.crs
        ref_shape = (ref.height, ref.width)

    with rasterio.open(src_path) as src:
        out_array = np.full(ref_shape, nodata, dtype=dtype)
        reproject(
            source=rasterio.band(src, 1),
            destination=out_array,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_transform, dst_crs=ref_crs,
            resampling=resampling,
            dst_nodata=nodata,
        )

    ref_profile.update(dtype=dtype, count=1, nodata=nodata, compress='lzw')
    with rasterio.open(out_path, 'w', **ref_profile) as dst:
        dst.write(out_array, 1)
    return out_path


def rasterize_vector_to_reference(gdf, ref_path, out_path,
                                    value=1, nodata=0, dtype='int8'):
    """Rasterize a GeoDataFrame to match a reference raster's grid.

    All non-empty geometries are burned with `value`. Empty cells get `nodata`.
    """
    with rasterio.open(ref_path) as ref:
        out_shape = (ref.height, ref.width)
        out_transform = ref.transform
        out_crs = ref.crs

    shapes = [
        (geom, value) for geom in gdf.geometry
        if geom is not None and not geom.is_empty
    ]
    raster = rio_rasterize(shapes, out_shape=out_shape,
                            transform=out_transform,
                            fill=nodata, dtype=dtype)

    profile = {
        'driver': 'GTiff', 'dtype': dtype,
        'width': out_shape[1], 'height': out_shape[0], 'count': 1,
        'crs': out_crs, 'transform': out_transform,
        'compress': 'lzw', 'nodata': nodata,
    }
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(raster, 1)
    return out_path

print("Helper functions loaded: get_alaska_boundary_3413, reproject_raster_to_3413, "
      "clip_raster_to_alaska, resample_to_reference_grid, rasterize_vector_to_reference")

## 2. LULC -- Dynamic World (Modal Land Cover)

In [ ]:
# Dynamic World v1 -- compute per-pixel modal (most common) class over 2023
# Classes: 0=water, 1=trees, 2=grass, 3=flooded_vegetation, 4=crops,
#          5=shrub_and_scrub, 6=built, 7=bare, 8=snow_and_ice

dw_collection = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterDate('2023-01-01', '2023-12-31')
    .filterBounds(alaska_boundary)
    .select('label')
)

# Compute per-pixel mode (most frequent class across the year)
lulc_modal = dw_collection.mode().clip(alaska_boundary).rename('lulc')

# Export
task = export_aligned_raster(
    image=lulc_modal.toInt8(),
    description='LULC_Alaska_Modal_2023',
    filename='lulc_alaska_modal',
)
export_tasks.append(task)

print("Dynamic World LULC classes:")
print("  0=water, 1=trees, 2=grass, 3=flooded_vegetation, 4=crops")
print("  5=shrub_and_scrub, 6=built, 7=bare, 8=snow_and_ice")

## 3. DEM & Slope -- FabDEM

In [ ]:
# FabDEM -- Forest And Buildings removed Copernicus DEM
# Source: https://gee-community-catalog.org/projects/fabdem/
fabdem = ee.ImageCollection('projects/sat-io/open-datasets/FABDEM').mosaic()

dem_alaska = fabdem.clip(alaska_boundary).rename('elevation')

# Export raw DEM (elevation in metres)
task_dem = export_aligned_raster(
    image=dem_alaska.toFloat(),
    description='DEM_Alaska_FabDEM',
    filename='dem_alaska',
)
export_tasks.append(task_dem)

# Compute slope in degrees
slope_alaska = ee.Terrain.slope(dem_alaska).rename('slope')

# Export slope
task_slope = export_aligned_raster(
    image=slope_alaska.toFloat(),
    description='Slope_Alaska_FabDEM',
    filename='slope_alaska',
)
export_tasks.append(task_slope)

print("DEM and slope exports started.")

## 4. Permafrost -- Pastick et al., 2015

Near-surface permafrost probability at 30m resolution from
[USGS ScienceBase](https://www.sciencebase.gov/catalog/item/5602ab5ae4b03bc34f5448b4)
(DOI: [10.5066/F7C53HX6](https://doi.org/10.5066/F7C53HX6)).

The download is a zip containing an ERDAS Imagine `.img` file (uint8).
Values 0-100 are permafrost probability (%); 101-105 are masked land
cover classes (water, ice/snow, developed, barren, cultivated).

Extract the `.img` file from the zip and upload to Google Drive.
Cell 4 loads it, reprojects to EPSG:3413, masks special codes, and
normalizes to 0-1 probability. Cell 4b resamples to 150m with
**bilinear interpolation** before reclassifying to zones.

In [ ]:
# Cell 4: Permafrost -- Pastick et al. 2015 (USGS ScienceBase)
# Source: https://www.sciencebase.gov/catalog/item/5602ab5ae4b03bc34f5448b4
# DOI: 10.5066/F7C53HX6
# Native resolution: 30m, CRS likely NAD83 Alaska Albers (EPSG:3338)
# Format: ERDAS Imagine .img (uint8, values 0-100 probability + 101-105 masked classes)
# The .img header references a companion .ige file for pixel data.
# Google Drive's FUSE mount can't resolve this cross-file reference,
# so we copy both files to local Colab storage before processing.

import glob

RASTER_DIR = os.environ.get('RASTER_DIR', './rasters')
os.makedirs(RASTER_DIR, exist_ok=True)

# Path to the .img file on Google Drive -- update to match your location
PASTICK_PATH = '/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_2.0/Pastick_permafrost_alaska.img'

if not os.path.exists(PASTICK_PATH):
    alt_path = PASTICK_PATH.rsplit('.', 1)[0] + '.tif'
    if os.path.exists(alt_path):
        PASTICK_PATH = alt_path
    else:
        raise FileNotFoundError(
            f"Pastick permafrost raster not found at: {PASTICK_PATH}\n"
            "Download from https://www.sciencebase.gov/catalog/item/5602ab5ae4b03bc34f5448b4\n"
            "Extract the .img file from the zip and upload to Google Drive."
        )

# Copy .img + .ige + sidecar files to local storage (avoids Drive FUSE issues)
LOCAL_DIR = '/content/pastick_local'
os.makedirs(LOCAL_DIR, exist_ok=True)
drive_dir = os.path.dirname(PASTICK_PATH)
stem = os.path.splitext(os.path.basename(PASTICK_PATH))[0]
for f in glob.glob(os.path.join(drive_dir, stem + '.*')):
    shutil.copy2(f, LOCAL_DIR)
    print(f"  Copied {os.path.basename(f)}")
# Also copy any .ige files (may have a different stem than the .img)
for f in glob.glob(os.path.join(drive_dir, '*.ige')):
    if not os.path.exists(os.path.join(LOCAL_DIR, os.path.basename(f))):
        shutil.copy2(f, LOCAL_DIR)
        print(f"  Copied {os.path.basename(f)}")

local_img = os.path.join(LOCAL_DIR, os.path.basename(PASTICK_PATH))
print(f"Using local copy: {local_img}")

# Reproject to EPSG:3413 (bilinear for continuous probability)
reproj_path = "/content/pastick_3413.tif"
clipped_path = "/content/pastick_3413_ak.tif"
print("Reprojecting Pastick permafrost to EPSG:3413...")
reproject_raster_to_3413(local_img, reproj_path, resampling=Resampling.bilinear)
clip_raster_to_alaska(reproj_path, clipped_path)

# Read probability values and normalise to 0-1 if needed
prob_output = os.path.join(RASTER_DIR, 'permafrost_prob_alaska.tif')
with rasterio.open(clipped_path) as src:
    data = src.read(1).astype(np.float32)
    src_nodata = src.nodata
    nodata_mask = np.isnan(data)
    if src_nodata is not None:
        nodata_mask |= (data == src_nodata)

    # Mask special codes (101-105: open water, ice/snow, developed, barren, cultivated)
    nodata_mask |= (data > 100)

    # Handle both 0-1 (float probability) and 0-100 (integer percent) ranges
    valid = data[~nodata_mask]
    if len(valid) > 0 and np.nanmax(valid) > 1.0:
        print(f"Probability range {np.nanmin(valid):.1f}-{np.nanmax(valid):.1f} detected, normalising to 0-1")
        data = data / 100.0

    data[nodata_mask] = -9999

    profile = src.profile.copy()
    profile.update(dtype='float32', nodata=-9999, compress='lzw', driver='GTiff')
    with rasterio.open(prob_output, 'w', **profile) as dst:
        dst.write(data, 1)

print(f"Permafrost probability saved: {prob_output}")
print("(At native ~30m resolution -- run Cell 4b to resample to 150m and reclassify)")

# Sanity check
with rasterio.open(prob_output) as src:
    d = src.read(1)
    valid = d[d != -9999]
    print(f"Shape: {src.shape}  CRS: {src.crs}  Res: {src.res[0]:.0f}m  Valid: {len(valid):,}")
    print(f"Probability  min={np.nanmin(valid):.3f}  mean={np.nanmean(valid):.3f}  max={np.nanmax(valid):.3f}")

## 4b. Resample Permafrost Probability to 150m, then Reclassify to Zones

Resamples the raw probability surface (float32, ~30m) to the 150m
reference grid using **bilinear interpolation** (smooth, continuous
data), then reclassifies into zone codes 0-3.

This order (resample-then-reclassify) eliminates the block artifacts
that occur when nearest-neighbor is applied to integer zone codes.

In [ ]:
# Cell 4b: Resample probability to 150m (bilinear), then reclassify to zones
# Run AFTER Cell 4 and AFTER lulc_alaska_modal.tif has been downloaded.

RASTER_DIR = os.environ.get('RASTER_DIR', './rasters')
ref_path  = os.path.join(RASTER_DIR, 'lulc_alaska_modal.tif')
prob_path = os.path.join(RASTER_DIR, 'permafrost_prob_alaska.tif')
out_path  = os.path.join(RASTER_DIR, 'permafrost_alaska.tif')
tmp_path  = os.path.join(RASTER_DIR, 'permafrost_prob_150m.tif')

if not os.path.exists(ref_path):
    raise FileNotFoundError(f"Reference raster not found: {ref_path}\nDownload lulc_alaska_modal.tif from Google Drive first.")
if not os.path.exists(prob_path):
    raise FileNotFoundError(f"Permafrost probability raster not found: {prob_path}\nRun Cell 4 first.")

# Step 1: Resample probability (continuous float) with bilinear interpolation
resample_to_reference_grid(prob_path, ref_path, tmp_path,
                            dtype='float32', resampling=Resampling.bilinear, nodata=-9999)
print("Resampled probability to 150m (bilinear)")

# Step 2: Reclassify resampled probability into zones 0-3
with rasterio.open(tmp_path) as src:
    prob = src.read(1).astype(np.float32)
    ref_profile = src.profile.copy()

nodata_mask = (prob == -9999) | np.isnan(prob)
zones = np.zeros(prob.shape, dtype=np.int8)
zones[(prob >= 0.1) & (prob < 0.5)] = 1   # sporadic
zones[(prob >= 0.5) & (prob < 0.9)] = 2   # discontinuous
zones[prob >= 0.9]                   = 3   # continuous
zones[nodata_mask]                   = -1  # nodata

ref_profile.update(dtype='int8', nodata=-1, compress='lzw')
with rasterio.open(out_path, 'w', **ref_profile) as dst:
    dst.write(zones, 1)

# Clean up intermediate file
os.remove(tmp_path)
print(f"Permafrost zones saved: {out_path}")

# Alignment verification + zone distribution
with rasterio.open(out_path) as pf, rasterio.open(ref_path) as ref:
    ok = pf.shape == ref.shape and pf.crs == ref.crs and pf.transform == ref.transform
    print(f"Alignment: {'PASS' if ok else 'FAIL'}")

with rasterio.open(out_path) as src:
    d = src.read(1); valid = d[d != -1]
    for zone, label in {0:'none', 1:'sporadic', 2:'discontinuous', 3:'continuous'}.items():
        pct = 100 * np.sum(valid == zone) / len(valid)
        print(f"  Zone {zone} ({label:>14s}): {pct:5.1f}%")

## 5. Roads -- GRIP4 (Presence)

In [ ]:
# GRIP4 -- Global Roads Inventory Project (version 4)
# Source: projects/sat-io/open-datasets/GRIP4/North-America

grip4_na = ee.FeatureCollection(
    'projects/sat-io/open-datasets/GRIP4/North-America'
).filterBounds(alaska_boundary)

print(f"GRIP4 North America features in Alaska: {grip4_na.size().getInfo()}")

roads_presence = (
    ee.Image(0)
    .byte()
    .paint(grip4_na, 1)
    .clip(alaska_boundary)
    .rename('road_presence')
)

print("Output: 1 = road present, 0 = no road")

task = export_aligned_raster(
    image=roads_presence.unmask(0).toInt8(),
    description='Roads_Presence_Alaska_GRIP4',
    filename='roads_presence_alaska',
)
export_tasks.append(task)

## 6. Navigable Waterways — NHD polygons + USACE NWN filter

Two-step approach to get true river width for navigable waterways:

1. **Cell 6** exports NHD waterbody and area polygon features from GEE as
   GeoJSON tables to Drive. These provide polygon geometry (true width) for
   lakes, wide rivers, and estuaries.

2. **Cell 6b** loads the NHD GeoJSON + the USACE NWN shapefile locally and
   performs a vector spatial join (`gpd.sjoin`) to keep only NHD polygons
   that intersect a navigable NWN line. NWN lines are unioned in as fallback
   for narrow navigable channels without NHD polygon geometry.

**Data sources:**
- [NHD (sat-io GEE catalog)](https://gee-community-catalog.org/projects/nhd/) — polygon geometry
- [USACE NWN](https://geospatial-usace.opendata.arcgis.com/datasets/ace7645d305647448a84492a3b909d48_1) — navigability authority (manual download)

In [ ]:
# Cell 6: Export NHD waterbody + area polygons for Alaska from GEE
# Source: projects/sat-io/open-datasets/NHD/NHD_AK/
# These provide true polygon width for rivers, lakes, and estuaries.
# Cell 6b will spatial-join them with NWN lines to filter for navigable only.

nhd_wb = ee.FeatureCollection(
    'projects/sat-io/open-datasets/NHD/NHD_AK/NHDWaterbody'
).filterBounds(alaska_boundary)

nhd_area = ee.FeatureCollection(
    'projects/sat-io/open-datasets/NHD/NHD_AK/NHDArea'
).filterBounds(alaska_boundary)

print(f"NHD Waterbody features: {nhd_wb.size().getInfo():,}")
print(f"NHD Area features:      {nhd_area.size().getInfo():,}")

# Export both as GeoJSON tables to Drive
for desc, fc_ee, fname in [
    ('NHD_Waterbody_AK', nhd_wb, 'nhd_waterbody_ak'),
    ('NHD_Area_AK', nhd_area, 'nhd_area_ak'),
]:
    task = ee.batch.Export.table.toDrive(
        collection=fc_ee,
        description=desc,
        folder=DRIVE_FOLDER,
        fileNamePrefix=fname,
        fileFormat='GeoJSON',
    )
    task.start()
    export_tasks.append(task)
    print(f"Table export started: {desc} -> {DRIVE_FOLDER}/{fname}.geojson")

## 6b. Combine NHD Polygons + NWN Navigability Filter

Performs a vector spatial join (`gpd.sjoin`) between NHD water polygons
(true width) and NWN navigable lines (navigability authority). Only NHD
features that intersect an NWN line are kept. NWN lines are unioned in
as fallback geometry for narrow navigable channels without NHD polygons.

Run AFTER both the NHD GeoJSON exports (Cell 6) are downloaded from Drive
and the LULC reference raster is available.

In [ ]:
# Cell 6b: Spatial join NHD polygons x NWN lines -> rasterize navigable waterways
# Run AFTER Cell 6 GeoJSON exports are downloaded from Drive.

RASTER_DIR = os.environ.get('RASTER_DIR', './rasters')
REFERENCE_RASTER = os.path.join(RASTER_DIR, 'lulc_alaska_modal.tif')
OUTPUT_RIVERS = os.path.join(RASTER_DIR, 'rivers_alaska.tif')

# Paths to GEE-exported NHD GeoJSON (downloaded from Drive to DRIVE_FOLDER)
DRIVE_PATH = '/content/drive/My Drive/' + DRIVE_FOLDER
NHD_WB_PATH = os.path.join(DRIVE_PATH, 'nhd_waterbody_ak.geojson')
NHD_AREA_PATH = os.path.join(DRIVE_PATH, 'nhd_area_ak.geojson')

# NWN shapefile (same path as before)
NWN_ZIP_PATH = '/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/NWN_Waterway_Network_Lines.zip'

# --- 1. Load NWN lines ---
print("Loading NWN lines...")
os.makedirs('/content/nwn', exist_ok=True)
with zipfile.ZipFile(NWN_ZIP_PATH, 'r') as z:
    z.extractall('/content/nwn')
shp_path = next((os.path.join(r, f) for r, _, fs in os.walk('/content/nwn')
                 for f in fs if f.endswith('.shp')), None)
if shp_path is None:
    raise FileNotFoundError(f"No .shp file found in {NWN_ZIP_PATH}")

nwn = gpd.read_file(shp_path).to_crs('EPSG:3413')
ak_gdf = get_alaska_boundary_3413()
nwn_ak = gpd.clip(nwn, ak_gdf)
print(f"NWN lines: {len(nwn_ak):,} features")

# --- 2. Load NHD polygons ---
print("Loading NHD polygons from GEE export...")
nhd_parts = []
for path, label in [(NHD_WB_PATH, 'Waterbody'), (NHD_AREA_PATH, 'Area')]:
    if os.path.exists(path):
        gdf = gpd.read_file(path)
        if gdf.crs is None or str(gdf.crs) != 'EPSG:3413':
            gdf = gdf.to_crs('EPSG:3413')
        print(f"  NHD {label}: {len(gdf):,} features")
        nhd_parts.append(gdf)
    else:
        print(f"  WARNING: {path} not found — skipping NHD {label}")

if not nhd_parts:
    raise FileNotFoundError(
        "No NHD GeoJSON files found. Run Cell 6, wait for GEE export, "
        "then download the GeoJSON files from Google Drive."
    )

nhd = pd.concat(nhd_parts, ignore_index=True)
nhd = gpd.GeoDataFrame(nhd, crs='EPSG:3413')

# Keep only polygon/multipolygon geometries
nhd = nhd[nhd.geometry.type.isin(['Polygon', 'MultiPolygon'])]
print(f"NHD polygons (combined): {len(nhd):,}")

# --- 3. Spatial join: keep NHD polygons that intersect NWN lines ---
print("Spatial join: NHD polygons x NWN lines (intersects)...")
navigable_nhd = gpd.sjoin(nhd, nwn_ak, predicate='intersects', how='inner')
navigable_nhd = navigable_nhd.drop_duplicates(subset='geometry')
print(f"Navigable NHD polygons: {len(navigable_nhd):,}")
print(f"  (filtered out {len(nhd) - len(navigable_nhd):,} non-navigable water bodies)")

# --- 4. Union navigable NHD polygons + NWN lines (fallback) ---
combined = pd.concat([
    navigable_nhd[['geometry']],
    nwn_ak[['geometry']],
], ignore_index=True)
combined = gpd.GeoDataFrame(combined, crs='EPSG:3413')
print(f"Combined features for rasterization: {len(combined):,}")

# --- 5. Rasterize to reference grid ---
if not os.path.exists(REFERENCE_RASTER):
    raise FileNotFoundError(f"Reference raster not found: {REFERENCE_RASTER}")
rasterize_vector_to_reference(combined, REFERENCE_RASTER, OUTPUT_RIVERS,
                                value=1, nodata=0, dtype='int8')

# --- 6. Summary ---
with rasterio.open(OUTPUT_RIVERS) as src:
    raster = src.read(1)
print(f"\nSaved: {OUTPUT_RIVERS}")
print(f"  Navigable pixels: {np.sum(raster == 1):,}")
print(f"  Non-waterway:     {np.sum(raster == 0):,}")

## 6c. Sea Ice Concentration -- NOAA/NSIDC CDR

Climatological mean winter sea ice concentration from the NOAA/NSIDC
Climate Data Record (CDR) of passive microwave sea ice concentration,
version 4 (G02202).

Computes the mean concentration over November-April for 2010-2024,
giving a 15-year winter climatology. Exported at 150m (resampled from
native 25km) as a float32 percentage (0-100%). Land cells are nodata.

This raster is used by `classify_water_type()` in `friction_surface.py`
to distinguish seasonally frozen (>70%), marginal (20-70%), and open
(<20%) water pixels.

In [ ]:
# Cell 6c: Sea ice concentration -- NOAA/NSIDC CDR v4 (G02202)
# Climatological winter mean (Nov-Apr, 2010-2024)
# Native resolution: 25km polar stereographic
# Output: float32 0-100% concentration, nodata over land

sea_ice = (
    ee.ImageCollection('NOAA/G02202')
    .filter(ee.Filter.calendarRange(11, 4, 'month'))  # Nov-Apr (winter)
    .filterDate('2010-01-01', '2024-12-31')
    .select('goddard_merged_seaice_conc')
)

print(f"Sea ice images (Nov-Apr, 2010-2024): {sea_ice.size().getInfo()}")

# Mean concentration over the 15-year winter window
sea_ice_mean = sea_ice.mean().clip(alaska_boundary).rename('sea_ice_conc')

# Scale from fraction (0-1) to percentage (0-100) if needed
# G02202 stores concentration as 0-1 fraction
sea_ice_pct = sea_ice_mean.multiply(100.0)

# Export at 150m (resampled from 25km via GEE's bilinear default)
task = export_aligned_raster(
    image=sea_ice_pct.toFloat(),
    description='Sea_Ice_Concentration_Alaska',
    filename='sea_ice_concentration_alaska',
)
export_tasks.append(task)

print("Sea ice concentration export started.")
print("Thresholds: >70% = seasonally frozen, 20-70% = marginal, <20% = open water")

## 6d. Region Raster -- Alaska Energy Authority Regions

Rasterizes the Alaska Energy Authority (AEA) energy region boundaries
from `Alaska_Energy_Authority_Library.shp` to the 150m reference grid.

Each pixel gets an integer region ID (1-10). The mapping from ID to
region name is stored in `friction_config.REGION_IDS`. Used by
`build_friction_barge_seasonal()` to apply regional seasonal overrides
(e.g., Southeast stays navigable in winter while Interior rivers freeze).

In [ ]:
# Cell 6d: Region raster -- Alaska Energy Authority regions
# Rasterize AEA region boundaries to the 150m reference grid.
# Output: int8 region IDs matching friction_config.REGION_IDS.

import friction_config as fc

RASTER_DIR = os.environ.get('RASTER_DIR', './rasters')
REFERENCE_RASTER = os.path.join(RASTER_DIR, 'lulc_alaska_modal.tif')
OUTPUT_REGIONS = os.path.join(RASTER_DIR, 'regions_alaska.tif')

# Path to AEA region shapefile -- update to match your Drive location
AEA_REGIONS_PATH = '/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/Alaska_Energy_Authority_Library.shp'

if not os.path.exists(AEA_REGIONS_PATH):
    raise FileNotFoundError(
        f"AEA region shapefile not found at: {AEA_REGIONS_PATH}\n"
        "Update AEA_REGIONS_PATH to your shapefile location."
    )
if not os.path.exists(REFERENCE_RASTER):
    raise FileNotFoundError(
        f"Reference raster not found: {REFERENCE_RASTER}\n"
        "Download lulc_alaska_modal.tif from Google Drive first."
    )

# Load and reproject to EPSG:3413
regions_gdf = gpd.read_file(AEA_REGIONS_PATH)
if regions_gdf.crs is None or str(regions_gdf.crs) != 'EPSG:3413':
    regions_gdf = regions_gdf.to_crs('EPSG:3413')

# Build inverse mapping: region name -> integer ID
name_to_id = {name: rid for rid, name in fc.REGION_IDS.items()}

# Identify the column containing region names
name_col = None
for col in ['NAME', 'Name', 'name', 'REGION', 'Region']:
    if col in regions_gdf.columns:
        name_col = col
        break
if name_col is None:
    raise ValueError(f"No region name column found. Columns: {list(regions_gdf.columns)}")

print(f"Region name column: {name_col}")
print(f"Unique regions in shapefile: {sorted(regions_gdf[name_col].unique())}")

# Assign integer IDs; warn about unmapped regions
regions_gdf['region_id'] = regions_gdf[name_col].map(name_to_id)
unmapped = regions_gdf[regions_gdf['region_id'].isna()][name_col].unique()
if len(unmapped) > 0:
    print(f"WARNING: unmapped regions (will be nodata): {list(unmapped)}")
regions_gdf = regions_gdf.dropna(subset=['region_id'])
regions_gdf['region_id'] = regions_gdf['region_id'].astype(int)

# Rasterize each region with its ID
with rasterio.open(REFERENCE_RASTER) as ref:
    out_shape = (ref.height, ref.width)
    out_transform = ref.transform
    out_crs = ref.crs

shapes = [(geom, rid) for geom, rid in
          zip(regions_gdf.geometry, regions_gdf['region_id'])
          if geom is not None and not geom.is_empty]

region_raster = rio_rasterize(shapes, out_shape=out_shape,
                               transform=out_transform,
                               fill=0, dtype='int8')

profile = {
    'driver': 'GTiff', 'dtype': 'int8',
    'width': out_shape[1], 'height': out_shape[0], 'count': 1,
    'crs': out_crs, 'transform': out_transform,
    'compress': 'lzw', 'nodata': 0,
}
with rasterio.open(OUTPUT_REGIONS, 'w', **profile) as dst:
    dst.write(region_raster, 1)

print(f"\nSaved: {OUTPUT_REGIONS}")
print("Region pixel counts:")
for rid, rname in sorted(fc.REGION_IDS.items()):
    count = int(np.sum(region_raster == rid))
    print(f"  {rid:2d} ({rname:>25s}): {count:>10,} pixels")

## 7. Airports -- OurAirports

Airport data is downloaded manually from [OurAirports](https://ourairports.com/data/)
(direct CSV: https://davidmegginson.github.io/ourairports-data/airports.csv)
and placed on Google Drive.

OurAirports is a free, community-maintained dataset that includes large, medium,
and small airports, heliports, and seaplane bases worldwide. This is significantly
more comprehensive than the FAA/OpenFlights data for rural Alaska, where small
airstrips and heliports are critical for fuel delivery to remote communities.

This cell loads `airports.csv`, filters to Alaska (`iso_region == 'US-AK'`),
reprojects from WGS84 to EPSG:3413, and exports as GeoJSON.

In [ ]:
# OurAirports -- load Alaska airports from CSV on Google Drive
# Source: https://ourairports.com/data/
# Direct CSV: https://davidmegginson.github.io/ourairports-data/airports.csv
# Download once, upload to Drive, update AIRPORTS_CSV_PATH below.

AIRPORTS_CSV_PATH = '/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/airports.csv'

if not os.path.exists(AIRPORTS_CSV_PATH):
    raise FileNotFoundError(
        f"OurAirports CSV not found at: {AIRPORTS_CSV_PATH}\n"
        "Download from https://davidmegginson.github.io/ourairports-data/airports.csv\n"
        "and upload to your Google Drive."
    )

# Load the CSV
airports_df = pd.read_csv(AIRPORTS_CSV_PATH)
print(f"Loaded {len(airports_df):,} airports worldwide")
print(f"Columns: {list(airports_df.columns)}")

# Filter to Alaska
ak_airports = airports_df[airports_df['iso_region'] == 'US-AK'].copy()
print(f"\nAlaska airports: {len(ak_airports):,}")

# Show type distribution so the user can see what's included
print("\nAirport type distribution:")
print(ak_airports['type'].value_counts())

# Drop any rows missing coordinates
ak_airports = ak_airports.dropna(subset=['latitude_deg', 'longitude_deg'])
print(f"\nAirports with valid coordinates: {len(ak_airports):,}")

# Build GeoDataFrame
airports_gdf = gpd.GeoDataFrame(
    ak_airports,
    geometry=gpd.points_from_xy(
        ak_airports['longitude_deg'],
        ak_airports['latitude_deg'],
    ),
    crs='EPSG:4326',
)

# Reproject to EPSG:3413
airports_3413 = airports_gdf.to_crs('EPSG:3413')
print(f"\nReprojected to EPSG:3413")

# Export as GeoJSON
output_dir = os.getenv('VECTOR_DIR', './vectors')
os.makedirs(output_dir, exist_ok=True)
airports_path = os.path.join(output_dir, 'airports_alaska.geojson')
airports_3413.to_file(airports_path, driver='GeoJSON')
print(f"\nAirports exported: {airports_path} ({len(airports_3413):,} features)")

## 8. Ports -- USACE NWN + AK DOT&PF (merged)

Merges two port datasets into a single source for the friction surface:

1. **USACE National Waterway Network (NWN) node layer** -- 41 Alaska port
   features with `PORT_ID` classification (C = coastal, I = inland/river).
   Fetched live from the USACE FeatureServer query API.

2. **AK DOT&PF Ports_and_Harbors** -- 147 Alaska state-maintained ports and
   harbors. Extracted from a shapefile zip on Google Drive.

The two are deduped by 1 km proximity (AK DOT features within 1 km of a
USACE feature are dropped as duplicates, keeping the USACE record for its
coastal/river classification).

Each merged port has:
- `port_class`: "port" or "beach_landing"
- `port_type`: "coastal", "inland_river", or "unknown"
- `source`: "USACE_NWN" or "AKDOT"
- `nwn_port_id`: USACE port ID (null for AK DOT records)

In [ ]:
# Ports -- merge USACE NWN nodes + AK DOT&PF Ports_and_Harbors
# USACE NWN provides coastal/inland_river classification; AK DOT provides
# broader Alaska small-port coverage. We dedupe by proximity (~1 km) so
# the same physical port doesn't appear twice.

import requests

# --- 1. Fetch USACE NWN nodes with PORT_ID set (41 Alaska ports) ---
NWN_NODE_URL = (
    "https://services7.arcgis.com/n1YM8pTrFmm7L4hs/arcgis/rest/services/"
    "Waterway_Networks/FeatureServer/0/query"
)
params = {
    "where": "STATE = 'AK' AND PORT_ID IS NOT NULL",
    "outFields": "*",
    "outSR": "4326",
    "f": "geojson",
    "resultRecordCount": 2000,
}

print("Fetching USACE NWN Alaska port nodes...")
resp = requests.get(NWN_NODE_URL, params=params, timeout=60)
resp.raise_for_status()
nwn_features = resp.json()

nwn_gdf = gpd.GeoDataFrame.from_features(
    nwn_features["features"], crs="EPSG:4326"
)
print(f"Loaded {len(nwn_gdf)} USACE NWN Alaska port nodes")
print(f"Columns: {list(nwn_gdf.columns)}")


# Classify coastal vs inland_river using PORT_ID prefix
def classify_nwn_port_type(pid):
    if pd.isna(pid):
        return "unknown"
    s = str(pid).strip().upper()
    if s.startswith("C"):
        return "coastal"
    if s.startswith("I"):
        return "inland_river"
    return "unknown"


nwn_gdf["port_type"] = nwn_gdf["PORT_ID"].apply(classify_nwn_port_type)
nwn_gdf["port_class"] = "port"  # USACE tracks commercial/deep-water ports
nwn_gdf["source"] = "USACE_NWN"

nwn_keep = nwn_gdf[["geometry", "port_class", "port_type", "source", "PORT_ID"]].copy()
nwn_keep = nwn_keep.rename(columns={"PORT_ID": "nwn_port_id"})

# --- 2. Load AK DOT&PF ports (existing workflow) ---
with zipfile.ZipFile(
    '/content/drive/My Drive/Masters/DOE-MAS/Version of Code/GDB_FL_1.0/codes/AK_Ports_and_Harbors.zip',
    'r',
) as z:
    z.extractall('/content/ak_ports_harbors')
PORTS_PATH = '/content/ak_ports_harbors/Ports_and_Harbors.shp'
if not os.path.exists(PORTS_PATH):
    raise FileNotFoundError(f"AK DOT port shapefile not found at: {PORTS_PATH}")

akdot_gdf = gpd.read_file(PORTS_PATH)
print(f"\nLoaded {len(akdot_gdf)} AK DOT&PF ports")

type_col = None
for col in ['TYPE', 'Type', 'type', 'PORT_TYPE', 'PortType', 'FACILITY',
            'FacilityTy', 'Facility_T', 'Facility', 'CLASS', 'Class']:
    if col in akdot_gdf.columns:
        type_col = col
        break

beach_keywords = ['beach', 'landing', 'lighter', 'barge landing', 'seasonal', 'small boat']


def classify_akdot_port(val):
    if pd.isna(val):
        return 'port'
    val_lower = str(val).lower().strip()
    if any(k in val_lower for k in beach_keywords):
        return 'beach_landing'
    return 'port'


if type_col is not None:
    akdot_gdf['port_class'] = akdot_gdf[type_col].apply(classify_akdot_port)
else:
    akdot_gdf['port_class'] = 'port'

akdot_gdf['port_type'] = 'unknown'  # AK DOT doesn't classify coastal/river
akdot_gdf['source'] = 'AKDOT'
akdot_gdf['nwn_port_id'] = None

akdot_keep = akdot_gdf[['geometry', 'port_class', 'port_type', 'source', 'nwn_port_id']].copy()

# --- 3. Reproject both to EPSG:3413 ---
nwn_3413 = nwn_keep.to_crs(TARGET_CRS)
akdot_3413 = akdot_keep.to_crs(TARGET_CRS)
print(f"\nReprojected both datasets to {TARGET_CRS}")

# --- 4. Dedupe AK DOT ports that are within 1 km of a USACE port ---
DEDUPE_RADIUS_M = 1000  # 1 km

nwn_buffer = nwn_3413.copy()
nwn_buffer['geometry'] = nwn_buffer.geometry.buffer(DEDUPE_RADIUS_M)
nwn_union = nwn_buffer.geometry.unary_union

akdot_keep_mask = ~akdot_3413.geometry.within(nwn_union)
akdot_unique = akdot_3413[akdot_keep_mask].copy()
akdot_dropped = (~akdot_keep_mask).sum()
print(f"\nDedupe: dropped {akdot_dropped} AK DOT ports within {DEDUPE_RADIUS_M}m of a USACE port")
print(f"  USACE ports kept:  {len(nwn_3413)}")
print(f"  AK DOT ports kept: {len(akdot_unique)}")

# --- 5. Union and save ---
merged = pd.concat([nwn_3413, akdot_unique], ignore_index=True)
merged_gdf = gpd.GeoDataFrame(merged, crs=TARGET_CRS)

print(f"\nTotal merged ports: {len(merged_gdf)}")
print("By port_class:")
print(merged_gdf['port_class'].value_counts())
print("By port_type:")
print(merged_gdf['port_type'].value_counts())
print("By source:")
print(merged_gdf['source'].value_counts())

output_dir = os.getenv('VECTOR_DIR', './vectors')
os.makedirs(output_dir, exist_ok=True)
ports_output = os.path.join(output_dir, 'ports_alaska.geojson')
merged_gdf.to_file(ports_output, driver='GeoJSON')
print(f"\nPorts exported: {ports_output}")

## 9. Facilities -- Bulk Fuel Sites

In [ ]:
# Bulk Fuel Facility Sites -- reproject from CSV to EPSG:3413
csv_path = '../Utilities_Bulk_Fuel_Inventory.csv'
if not os.path.exists(csv_path):
    csv_path = 'Utilities_Bulk_Fuel_Inventory.csv'

bulk_fuel = pd.read_csv(csv_path, usecols=[
    'ASTFacilityID', 'ASTFacilityLongitude', 'ASTFacilityLatitude',
    'CommunityName', 'Delivery_method'
])
bulk_fuel = bulk_fuel.dropna(subset=['ASTFacilityLongitude', 'ASTFacilityLatitude'])

facilities_gdf = gpd.GeoDataFrame(bulk_fuel,
    geometry=gpd.points_from_xy(bulk_fuel['ASTFacilityLongitude'],
                                 bulk_fuel['ASTFacilityLatitude']),
    crs='EPSG:4326')

facilities_3413 = facilities_gdf.to_crs('EPSG:3413')
output_dir = os.getenv('VECTOR_DIR', './vectors')
os.makedirs(output_dir, exist_ok=True)
facilities_path = os.path.join(output_dir, 'facilities_alaska.geojson')
facilities_3413.to_file(facilities_path, driver='GeoJSON')
print(f"Facilities saved: {facilities_path} ({len(facilities_3413)} sites)")

## 10. Monitor GEE Export Tasks & Alignment Verification

In [ ]:
# Monitor export tasks
import time

def monitor_tasks(tasks, poll_interval=30):
    print(f"Monitoring {len(tasks)} export tasks...")
    while True:
        statuses = {}
        for t in tasks:
            status = t.status()
            statuses[status['description']] = status['state']
        completed = sum(1 for s in statuses.values() if s == 'COMPLETED')
        failed = sum(1 for s in statuses.values() if s == 'FAILED')
        running = sum(1 for s in statuses.values() if s in ('RUNNING', 'READY'))
        print(f"\r  Completed: {completed} | Running: {running} | Failed: {failed}", end='')
        if running == 0:
            print()
            break
        time.sleep(poll_interval)
    print("\nFinal status:")
    for desc, state in statuses.items():
        print(f"  {desc}: {state}")

# Uncomment to monitor:
# monitor_tasks(export_tasks)

In [ ]:
# Alignment Verification
# Run this AFTER downloading exported rasters from Google Drive to the rasters/ directory

import os
import numpy as np
import rasterio

raster_dir = os.getenv('RASTER_DIR', './rasters')
raster_files = {
    'LULC': os.path.join(raster_dir, 'lulc_alaska_modal.tif'),
    'Slope': os.path.join(raster_dir, 'slope_alaska.tif'),
    'DEM': os.path.join(raster_dir, 'dem_alaska.tif'),
    'Permafrost': os.path.join(raster_dir, 'permafrost_alaska.tif'),
    'Roads Presence': os.path.join(raster_dir, 'roads_presence_alaska.tif'),
    'Rivers': os.path.join(raster_dir, 'rivers_alaska.tif'),
    'Sea Ice Conc.': os.path.join(raster_dir, 'sea_ice_concentration_alaska.tif'),
    'Regions': os.path.join(raster_dir, 'regions_alaska.tif'),
}

print(f"{'Layer':<20} {'Shape':<20} {'CRS':<15} {'Res (m)':<12} {'Dtype':<10} {'Min':<10} {'Max':<10}")
print("-" * 97)

reference_crs = reference_transform = reference_shape = None
all_aligned = True

for name, path in raster_files.items():
    if not os.path.exists(path):
        print(f"{name:<20} FILE NOT FOUND: {path}")
        all_aligned = False
        continue
    with rasterio.open(path) as src:
        data = src.read(1)
        crs, res, shape = str(src.crs), src.res, (src.height, src.width)
        dtype, transform = str(src.dtypes[0]), src.transform
        if reference_crs is None:
            reference_crs, reference_transform, reference_shape = crs, transform, shape
        else:
            if crs != reference_crs or shape != reference_shape or transform != reference_transform:
                all_aligned = False
        valid = data[data != src.nodata] if src.nodata else data
        vmin = f"{np.nanmin(valid):.1f}" if len(valid) > 0 else "N/A"
        vmax = f"{np.nanmax(valid):.1f}" if len(valid) > 0 else "N/A"
        print(f"{name:<20} {str(shape):<20} {crs:<15} {res[0]:<12.1f} {dtype:<10} {vmin:<10} {vmax:<10}")

print()
print("ALL RASTERS ALIGNED" if all_aligned else "WARNING: Raster alignment issues detected.")